# WakeRef — Fine-tuning Whisper

**Avant de commencer :** menu **Exécution → Modifier le type d'exécution → T4 GPU**.

Lance les cellules **dans l'ordre** (▶). Étapes : install → dataset → script → entraînement → téléchargement.


## 1. Installer les outils (~2 min)

In [ ]:
!pip -q install -U transformers datasets accelerate evaluate jiwer librosa soundfile


## 2. Envoyer le dataset (.zip) → décompresser → convertir en wav
Quand le bouton **« Choisir les fichiers »** apparaît, sélectionne les fichiers `wakeref-voix-dataset-*.zip`.

In [ ]:
from google.colab import files
files.upload()   # sélectionne TOUS les .zip (le tien + ceux des autres)
import zipfile, csv, os, glob, shutil
os.makedirs('ds/audio', exist_ok=True); rows = []; n = 0
for z in sorted(glob.glob('*.zip')):
    tmp = 'tmp_' + os.path.basename(z)
    with zipfile.ZipFile(z) as zf: zf.extractall(tmp)
    base = next(r for r, _, fs in os.walk(tmp) if 'metadata.csv' in fs)
    with open(os.path.join(base, 'metadata.csv')) as f:
        for row in csv.DictReader(f):
            n += 1; ext = os.path.splitext(row['file_name'])[1]
            shutil.copy(os.path.join(base, row['file_name']), f'ds/audio/{n:05d}{ext}')
            row['file_name'] = f'audio/{n:05d}{ext}'; rows.append(row)
with open('ds/metadata.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys()); w.writeheader(); w.writerows(rows)
!for f in ds/audio/*.webm; do ffmpeg -nostdin -loglevel error -y -i "$f" -ar 16000 -ac 1 "${f%.webm}.wav"; done
print(len(rows), 'clips fusionnes dans ds/')

## 3. Créer le script d'entraînement
Réponse attendue : `Writing finetune_whisper.py`.

In [ ]:
%%writefile finetune_whisper.py
#!/usr/bin/env python
# Fine-tune Whisper sur le dataset de dictée WakeRef (export .zip de /judge/voix).
#
# Cible d'entraînement = le NOM CANONIQUE du trick (colonne `transcription` du
# metadata.csv). Le modèle apprend audio → nom de trick directement : sur un set
# FERMÉ + une voix, c'est le plus direct, et le matcher de l'app reste un filet.
#
# L'audio est chargé À LA VOLÉE dans le collator (pas de pré-traitement de masse →
# pas de saturation mémoire / plantage `datasets.map`). Pensé pour Colab (T4 gratuit).
#
#   pip install -U transformers datasets accelerate evaluate jiwer librosa soundfile
#   python finetune_whisper.py --data ./ds --base openai/whisper-base --out ./whisper-wakeref
#
import argparse, os
from dataclasses import dataclass
from typing import Any
import pandas as pd
import librosa
import torch
from datasets import Dataset
from transformers import (WhisperProcessor, WhisperForConditionalGeneration,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments)
import evaluate

ap = argparse.ArgumentParser()
ap.add_argument("--data", required=True, help="dossier décompressé du zip (metadata.csv + audio/)")
ap.add_argument("--base", default="openai/whisper-base")
ap.add_argument("--out",  default="./whisper-wakeref")
ap.add_argument("--epochs", type=float, default=8.0)
ap.add_argument("--bs", type=int, default=16)
args = ap.parse_args()

# ── Données : on garde juste (chemin, texte) ; l'audio est lu dans le collator ──
df = pd.read_csv(os.path.join(args.data, "metadata.csv"))
df = df[df["transcription"].notna() & (df["transcription"].astype(str).str.strip() != "")]
df["path"] = df["file_name"].apply(lambda p: os.path.join(args.data, p))
print(f"{len(df)} clips · {df['transcription'].nunique()} tricks distincts")

ds = Dataset.from_pandas(df[["path", "transcription"]], preserve_index=False)
ds = ds.train_test_split(test_size=0.1, seed=42)

processor = WhisperProcessor.from_pretrained(args.base, language="french", task="transcribe")

def resolve(p):
    # Préfère un .wav 16 kHz pré-converti s'il existe (décodage webm fiable via ffmpeg).
    wav = os.path.splitext(p)[0] + ".wav"
    return wav if os.path.exists(wav) else p

# ── Collator : charge l'audio + calcule mel + tokenise, par batch ──────────────
@dataclass
class Collator:
    processor: Any
    def __call__(self, features):
        audios = [librosa.load(resolve(f["path"]), sr=16000, mono=True)[0] for f in features]
        feats = self.processor.feature_extractor(audios, sampling_rate=16000, return_tensors="pt")
        lab = self.processor.tokenizer([f["transcription"] for f in features], padding=True, return_tensors="pt")
        labels = lab["input_ids"].masked_fill(lab.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        return {"input_features": feats.input_features, "labels": labels}

# ── Modèle ────────────────────────────────────────────────────────────────────
model = WhisperForConditionalGeneration.from_pretrained(args.base)
model.generation_config.language = "french"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

metric = evaluate.load("wer")
def compute_metrics(p):
    pred = p.predictions
    pred[pred == -100] = processor.tokenizer.pad_token_id
    labels = p.label_ids
    labels[labels == -100] = processor.tokenizer.pad_token_id
    ps = processor.tokenizer.batch_decode(pred, skip_special_tokens=True)
    ls = processor.tokenizer.batch_decode(labels, skip_special_tokens=True)
    return {"wer": 100 * metric.compute(predictions=ps, references=ls)}

targs = Seq2SeqTrainingArguments(
    output_dir=args.out,
    per_device_train_batch_size=args.bs,
    per_device_eval_batch_size=args.bs,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    warmup_ratio=0.1,
    num_train_epochs=args.epochs,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    predict_with_generate=True,
    generation_max_length=64,
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    save_total_limit=1,            # ne garde qu'un checkpoint (sinon le dossier pèse des Go)
    remove_unused_columns=False,   # le collator a besoin des colonnes path/transcription
    report_to=[],
)

trainer = Seq2SeqTrainer(
    model=model, args=targs,
    train_dataset=ds["train"], eval_dataset=ds["test"],
    data_collator=Collator(processor), compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

trainer.train()
trainer.save_model(args.out)
processor.save_pretrained(args.out)
print(f"\n✓ modèle fine-tuné → {args.out}")


## 4. Entraîner (~20-40 min)
La **loss** doit descendre ; la **WER** (taux d'erreur) s'affiche à chaque epoch. Le meilleur modèle est gardé automatiquement.

In [ ]:
!python finetune_whisper.py --data ds --base openai/whisper-base --out whisper-wakeref


## 5. Récupérer le modèle (zip propre, ~290 Mo)
On retire les checkpoints d'entraînement (lourds, inutiles) avant de zipper.

In [ ]:
!cd whisper-wakeref && rm -rf checkpoint-* && cd .. && zip -qr whisper-wakeref.zip whisper-wakeref
from google.colab import files
files.download('whisper-wakeref.zip')
